In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv
import os

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages

from langgraph.checkpoint.memory import MemorySaver

from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage

# LOAD ENV VARIABLES
load_dotenv()

# OPENROUTER API KEY
OPENROUTER_API_KEY = os.getenv("ROUTER_API_TOKEN")

# LLM
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",

    api_key=OPENROUTER_API_KEY,

    base_url="https://openrouter.ai/api/v1"
)

# MEMORY
memory = MemorySaver()

# STATE
class State(TypedDict):
    messages: Annotated[list, add_messages]

# NODE
def chatbot(state: State):
    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }

# GRAPH
graph = StateGraph(State)

graph.add_node("chatbot", chatbot)

graph.add_edge(START, "chatbot")

# COMPILE GRAPH
app = graph.compile(
    checkpointer=memory
)

# THREAD CONFIG
config = {
    "configurable": {
        "thread_id": "user_1"
    }
}



c:\Users\manikandan.r\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Desktop\Hugging_face_api_final\env\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
c:\Users\manikandan.r\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Desktop\Hugging_face_api_final\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# CHAT LOOP
while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    result = app.invoke(
        {
            "messages": [
                HumanMessage(content=user_input)
            ]
        },
        config=config
    )

    print(
        "AI:",
        result["messages"][-1].content
    )

AI: Hi Manikandan! How can I assist you today?
AI: Your name is Manikandan. How can I help you today?
